# 02 - Product Feed Analysis

## Purpose

The purpose of this notebook is to analyze the Snickers Product Feed (`PP_Export`) used in the import workflow.

The analysis focuses on:

- XML file structure
- Root element and main record elements
- Number of products and variants
- Available product attributes
- Product and variant identifiers
- Missing values and data coverage
- Product descriptions and feature fields
- Image availability
- Potential source-specific issues affecting the ETL pipeline

## Imports

In [ ]:
import xml.etree.ElementTree as ET

## File Configuration

In [ ]:
SUPPLIER = "snickers"

PRODUCT_FEED_PATH = (
    f"../data/{SUPPLIER}/product_feeds/PP_Export_Snickers_018_sv.xml"
)

## Raw File Inspection

Inspect the raw XML file before parsing it. This helps identify:

- Encoding
- XML declaration
- Root structure
- Record element structure
- Namespaces
- Unexpected formatting issues

In [ ]:
with open(PRODUCT_FEED_PATH, encoding="utf-8") as file:
    for _ in range(5):
        print(repr(file.readline()))

## Load Dataset

In [ ]:
tree = ET.parse(PRODUCT_FEED_PATH)
root = tree.getroot()

## Dataset Overview

In [ ]:
root.tag

In [ ]:
product_elements = root.findall("ProductInfo")
number_of_records = len(product_elements)
number_of_records

In [ ]:
root[0].tag

In [ ]:
root[0].attrib

In [ ]:
[child.tag for child in root[0]]

## Identifier Analysis

In [ ]:
def get_text(element, tag):
    child = element.find(tag)
    return child.text.strip() if child is not None and child.text else None

sample_identifiers = [
    {
        "StockCode": get_text(product, "StockCode"),
        "ModelCode": get_text(product, "ModelCode"),
        "ColourCode": get_text(product, "ColourCode"),
        "Size": get_text(product, "Size"),
    }
    for product in product_elements[:10]
]

sample_identifiers

In [ ]:
stock_codes = [
    get_text(product, "StockCode")
    for product in product_elements
]

print("Records:", len(stock_codes))
print("Missing StockCode:", sum(code is None for code in stock_codes))
print("Duplicate StockCode:", len(stock_codes)- len(set(stock_codes)))

Observed:
- The Product Feed contains 29,606 `ProductInfo` records.
- `StockCode` is populated for all records.
- No duplicate `StockCode` values were identified.
- `StockCode` can therefore be treated as the unique variant-level identifier within the Product Feed.

In [ ]:
identifier_fields = ["ModelCode", "ColourCode", "Size"]

for field in identifier_fields:
    values = [get_text(product, field) for product in product_elements]

    print(f"{field}:")
    print(f" Missing: {sum(value is None for value in values)}")
    print(f" Unique: {len(set(value for value in values if value is not None))}")

Observed:
- `ModelCode`, `ColourCode` and `Size` are populated for all 29,606 Product Feed records.
- The feed contains 679 unique model codes, 106 unique colour codes and 149 unique size values.
- Together with the unique `StockCode`, these fields provide variant-level structure within the Product Feed.

## Size Analysis

In [ ]:
size_values = sorted({
    get_text(product, "Size")
    for product in product_elements
})

size_values

Observed:

- `Size` contains 149 distinct values in the Product Feed.
- The field contains several numeric size ranges as well as other encoded size values.
- Numeric size values must not be classified as standard or special based on the `Size` value alone.
- Classification of webshop-eligible and special-priced variants requires cross-source mapping with the Price List.

## Product Content Analysis

In [ ]:
content_fields = [
    "Intro",
    "TechnicalDescription",
    "Feature1",
    "Feature2",
    "Feature3",
    "Feature4",
    "Feature5",
    "Feature6",
    "Feature7",
]

for field in content_fields:
    values = [get_text(product, field) for product in product_elements]

    populated = sum(value is not None for value in values)

    print(f"{field}: {populated} / {len(product_elements)}")

Observed:
- `Intro` is populated for 29,569 of 29,606 Product Feed records.
- `TechnicalDescription` is populated for 26,046 records.
- `Feature1`–`Feature3` have near-complete coverage.
- Coverage decreases for later feature fields: `Feature4` and `Feature5` remain common, while `Feature6` is populated for 15,396 records.
- `Feature7` is rare and is populated for only 50 records.
- Description and feature fields are therefore optional at source level and must be handled without assuming that every field is populated.

## Image Coverage

In [ ]:
image_field = "This_x0020_is_x0020_the_x0020_main_x0020_image"

image_values = [
    get_text(product, image_field)
    for product in product_elements
]

print("With main image:", sum(value is not None for value in image_values))
print("Without main image:", sum(value is None for value in image_values))

Observed:
- 27,489 of 29,606 Product Feed records contain a main image.
- 2,117 records do not contain a main image.
- Main image availability is therefore not complete at source level and must be handled as optional data during transformation.

## EAN Analysis

In [ ]:
ean_values = [
    get_text(product, "EAN_text")
    for product in product_elements
]

populated_ean = [value for value in ean_values if value is not None]

print("With EAN:", len(populated_ean))
print("Without EAN:", len(ean_values) - len(populated_ean))
print("Duplicate EAN:", len(populated_ean) - len(set(populated_ean)))

Observed:
- `EAN_text` is populated for 29,563 of 29,606 Product Feed records.
- 43 records do not contain an EAN value.
- The Product Feed contains 12 EAN occurrences beyond the number of unique populated EAN values.
- EAN should therefore be treated as product metadata rather than a unique variant identifier.
- `StockCode` remains the unique variant-level identifier within the Product Feed.

## Findings

- The Product Feed contains 29,606 `ProductInfo` records.
- `StockCode` is populated and unique for all records and can be used as the variant-level identifier within the Product Feed.
- `ModelCode`, `ColourCode` and `Size` are populated for all records.
- The feed contains 679 unique model codes, 106 unique colour codes and 149 unique size values.
- The `Size` field contains several encoded and numeric size ranges. Whether a size variant should be included in the webshop cannot be determined from the `Size` field alone and requires comparison with the Price List.
- Product content is distributed across `Intro`, `TechnicalDescription` and `Feature1`–`Feature7`, with varying field coverage.
- `Intro` and the first feature fields have near-complete coverage, while later feature fields are progressively less populated.
- Main image coverage is incomplete: 27,489 records contain a main image and 2,117 do not.
- `EAN_text` is populated for 29,563 records, while 43 records have no EAN and duplicate EAN values exist.
- EAN should therefore not be used as the unique variant identifier; `StockCode` remains the reliable unique identifier within this source.

## Open Questions

- Does the Product Feed contain discontinued, historical or otherwise non-current product variants?
- What causes the 43 records without an EAN value?
- Why are some EAN values shared by multiple Product Feed records?
- Why do 2,117 records lack a main image?
- Which Product Feed fields should be retained for the webshop transformation?